<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/circuit_tracer_examples/attention_attribution_analysis.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

In [ ]:
# Uncomment to run installation steps if you do not have a development
# editable install and want to run this notebook in a fresh environment.
# %pip install uv
# %uv pip install --upgrade pip setuptools wheel && \
# %uv pip install 'git+https://github.com/speediedan/interpretune.git@main[examples]'
# %uv pip install --group git-deps
#
# NOTE: This cell is intentionally commented out. We will uncomment these
# install commands once we no longer need to preserve editable installs
# for active developer venvs.


# Attention-Aware Attribution

An attribution graph holds the attention patterns and the norm denominators constant by default. That is what
makes its edges exact direct effects in a linearized model, and it is also a blind spot: an effect that acts by
changing *where attention looks* is credited to no node, and an edge does not say which head carried it.

This notebook demonstrates the three attention-aware additions to the analysis surface, each through its
top-level wrapper, on one prompt whose answer is easy to state:

- **The two regimes:** `compute_attribution_graph` with patterns frozen (the default) and with patterns live
  (`freeze_attention=False`). A live-pattern edge is a first-order derivative, not an exact direct effect.
- **Head loadings:** `it.head_loadings(...)` splits the edges into one target across the heads of a layer.
- **Head conductance:** `it.head_conductance(...)` credits each head for the measured change of a target
  between two prompts of the same length, split into a pattern (QK) part and a value (OV) part.

Only the nnsight model-level adapter supports these reports for now. A backend without the
`head_attribution` capability is refused by name, and circuit-tracer refuses a non-nnsight backend at the call.


In [ ]:
# @title Imports { display-mode: "form" }
import json
from pprint import pformat

import torch

import interpretune as it
import interpretune.analysis  # registers the top-level op wrappers used below
from it_examples.seeds import ensure_local_seeds
from it_examples.utils.example_helpers import required_os_env
from interpretune import ITSession, ITSessionConfig
from interpretune.config import AnalysisCfg, init_analysis_cfgs

## Notebook Parameters

This cell contains parameters that can be injected by papermill during parameterized test runs.
`conductance_steps` is the step count of the path integral in the conductance section: a path that moves
sharply needs many more steps than the default, which is exactly what its completeness residual reports.


In [ ]:
# Parameters - These will be injected by papermill during parameterized test runs
core_log_dir = None  # Directory to save analysis logs (if None, a temp directory will be created)
conductance_steps = 32  # Path-integral steps for head_conductance; read its residual before trusting few steps
head_layers = [13, 17, 21, 25]  # Layers for the head-loadings table (None reports every layer)

In [ ]:
# @title Environment Setup { display-mode: "form" }
env_path: str | None = None  # set to '/full/path/to/.env' to override
os_env_reqs = None
assert required_os_env(env_path=env_path, env_reqs=os_env_reqs)

## Session Configuration

We load the `speediedan/rte` component configuration for Gemma-2-2b via `it.hub.load`, with the nnsight
backend and the Gemma Scope transcoder set, the same substrate as the
[CT analysis backend demo](ct_analysis_backend_demo.ipynb).


In [ ]:
ensure_local_seeds()  # idempotent, offline: seed publish sources -> local components cache
base_itdm_cfg, base_it_cfg, dm_cls, m_cls = it.hub.load("speediedan/rte", "rte_demo.gemma2.circuit_tracer")

base_it_cfg.circuit_tracer_cfg.backend = "nnsight"
base_it_cfg.circuit_tracer_cfg.transcoder_set = "gemma"
if core_log_dir:
    base_it_cfg.core_log_dir = core_log_dir
print(pformat(base_it_cfg.circuit_tracer_cfg))

session_cfg = ITSessionConfig(
    adapter_ctx=(it.Adapter.core, it.Adapter.nnsight, it.Adapter.circuit_tracer),
    datamodule_cfg=base_itdm_cfg,
    module_cfg=base_it_cfg,
    datamodule_cls=dm_cls,
    module_cls=m_cls,
)
it_session = ITSession(session_cfg)
it.it_init(**it_session)

module = it_session.module
tokenizer = module.replacement_model.tokenizer
module.analysis_cfg = AnalysisCfg(target_op=it.compute_attribution_graph, ignore_manual=True, save_tokens=False)
init_analysis_cfgs(module, [module.analysis_cfg])
print(f"Module type: {type(module).__name__}")
print("Session initialized successfully!")

## One Prompt, One Swap

The prompt is the capital prompt from the backend demo. The counterfactual swaps one city for another of the
same token length, which `head_conductance` requires: it integrates along the line between the two prompts'
embeddings, so the two must align position by position. Dallas and Denver sit in different states, so the
swap moves the Austin-versus-Denver logit difference by a measurable amount.


In [ ]:
prompt = "Fact: the capital of the state containing Dallas is"
swapped_prompt = "Fact: the capital of the state containing Denver is"

prompt_ids = tokenizer.encode(prompt)
swapped_ids = tokenizer.encode(swapped_prompt)
assert len(prompt_ids) == len(swapped_ids), "head_conductance needs two prompts of the same token length"
changed = [i for i, (a, b) in enumerate(zip(prompt_ids, swapped_ids, strict=True)) if a != b]
print(f"Prompt tokens: {[tokenizer.decode([t]) for t in prompt_ids]}")
print(
    f"Changed positions: {changed} ({[tokenizer.decode([prompt_ids[i]]) for i in changed]} -> "
    f"{[tokenizer.decode([swapped_ids[i]]) for i in changed]})"
)

austin_id = tokenizer.encode("▁Austin", add_special_tokens=False)[-1]
denver_id = tokenizer.encode("▁Denver", add_special_tokens=False)[-1]
print(f"Key tokens: Austin={austin_id}, Denver={denver_id}")

## The Two Regimes Disagreeing

The same graph computed twice: once with the default regime (patterns and norms frozen) and once with
patterns live. The regime is recorded on each graph, and the record matters: a graph whose record was lost
reads as frozen, which would mislabel a patterns-live graph with no error anywhere.

Under the live regime an edge is a first-order derivative of the target with respect to the source, read at
this prompt. It credits a source for changing where attention looks, at the price of exactness: the frozen
graph's edges are exact direct effects in the linearized model, and the live graph's are not.


In [ ]:
frozen = it.compute_attribution_graph(
    module, None, None, 0, prompt=prompt, max_n_logits=5, desired_logit_prob=0.5, verbose=False
)
live = it.compute_attribution_graph(
    module,
    None,
    None,
    0,
    prompt=prompt,
    max_n_logits=5,
    desired_logit_prob=0.5,
    verbose=False,
    freeze_attention=False,
)

print(f"Frozen regime: {json.loads(frozen.graph_attribution_regime_json)}")
print(f"Live regime:   {json.loads(live.graph_attribution_regime_json)}")
target_probs = [round(float(p), 4) for p in frozen.logit_probabilities]
print(f"Logit targets: {frozen.logit_target_tokens} (probabilities {target_probs})")

assert frozen.adjacency_matrix.shape == live.adjacency_matrix.shape
assert torch.equal(frozen.active_features, live.active_features), (
    "the two regimes must share one node set to compare edges"
)
delta = (live.adjacency_matrix - frozen.adjacency_matrix).abs()
print(f"Edges changed: {int((delta > 0).sum())} of {delta.numel()}; largest change {float(delta.max()):.4f}")
assert float(delta.max()) > 0, "positive control: the two regimes must actually disagree"

Rows of the adjacency matrix are target nodes and columns are sources, in the order
features, error nodes, token (embed) nodes, logit nodes. The table reads the Austin logit row under both
regimes, for the sources whose edge is largest in either one.


In [ ]:
def node_label(batch, idx):
    n_features = len(batch.active_features)
    n_pos = len(batch.input_tokens)
    n_layers = module.replacement_model.cfg.n_layers
    if idx < n_features:
        layer, pos, feat = (int(v) for v in batch.active_features[idx])
        return f"feature L{layer} pos{pos} #{feat}"
    idx -= n_features
    if idx < n_layers * n_pos:
        return f"error L{idx // n_pos} pos{idx % n_pos}"
    idx -= n_layers * n_pos
    if idx < n_pos:
        return f"token pos{idx} {tokenizer.decode([int(batch.input_tokens[idx])])!r}"
    idx -= n_pos
    return f"logit {batch.logit_target_tokens[idx]!r}"


target_ids = [int(t) for t in frozen.logit_target_ids]
assert austin_id in target_ids, f"Austin must be a logit target of the graph, got {frozen.logit_target_tokens}"
austin_row = frozen.adjacency_matrix.shape[0] - len(target_ids) + target_ids.index(austin_id)

strength = torch.maximum(frozen.adjacency_matrix[austin_row].abs(), live.adjacency_matrix[austin_row].abs())
top_sources = strength.topk(8).indices.tolist()
print(f"{'source into Austin':<28} {'frozen':>10} {'live':>10}")
for src in top_sources:
    print(
        f"{node_label(frozen, src):<28} {float(frozen.adjacency_matrix[austin_row, src]):>10.4f} "
        f"{float(live.adjacency_matrix[austin_row, src]):>10.4f}"
    )

## Head Loadings

`head_loadings` splits the edges into one target across the heads of each requested layer, plus a bypass
share for the part of those edges no head carries. Each number is a share: the head's part of the summed
edges into the target at that layer. That sum is the share's denominator, and it is the thing to look at
before quoting a share: where the summed edges at a layer are small, the same absolute attribution becomes
a large share, and small measurement differences move it a lot. The partition check beside the table is the
internal control: the head shares and the bypass must sum back to the whole, up to the reported partition
error.


In [ ]:
loadings = it.head_loadings(module, None, None, 0, prompt=prompt, head_target="▁Austin", head_layers=head_layers)

partitioned = loadings.head_loading_shares.sum(1) + loadings.head_loading_bypass_shares
print(f"Summed edges into the target (the shares' denominator): {float(loadings.head_loading_total):.4f}")
print(f"{'layer':>5} {'top head':>8} {'share':>8} {'bypass':>8} {'partition err':>13}")
for row, layer in enumerate(loadings.head_loading_layers.tolist()):
    shares = loadings.head_loading_shares[row]
    top_head = int(shares.abs().argmax())
    print(
        f"{layer:>5} {top_head:>8} {float(shares[top_head]):>8.3f} "
        f"{float(loadings.head_loading_bypass_shares[row]):>8.3f} "
        f"{float(loadings.head_loading_partition_errors[row]):>13.2e}"
    )

print(f"Partition check (shares + bypass, should read 1 per layer): {[round(float(v), 4) for v in partitioned]}")
assert bool(((partitioned - 1).abs() <= loadings.head_loading_partition_errors + 0.01).all()), (
    "head parts must partition the edges up to the reported partition error"
)
assert float(loadings.head_loading_shares.abs().max()) > 1e-3, "a report of zeros would also partition the edges"

## Head Conductance

Conductance answers a different question: not what the graph credits on one prompt, but which heads carry a
*measured change* between two prompts. It integrates each head's pattern (QK) and value (OV) contribution
along the straight path between the Dallas and Denver embeddings, for the Austin-versus-Denver logit
difference.

Read `head_conductance_completeness_residual` before any number below it. The residual is what the integral
failed to account for at each layer, and it falls quadratically with the step count when the path is merely
curved, so a large residual at the default step count is an instruction to raise `conductance_steps`, not a
result. `head_conductance_first_order` is the one-point linearization at the first prompt: where it disagrees
with the measured change, it is the case for integrating along the path at all.


In [ ]:
target_weights = torch.zeros(int(frozen.graph_vocab_size))
target_weights[austin_id], target_weights[denver_id] = 1.0, -1.0

conductance = it.head_conductance(
    module,
    None,
    None,
    0,
    prompt=prompt,
    counterfactual_prompt=swapped_prompt,
    target_weights=target_weights,
    n_steps=conductance_steps,
)

residual = conductance.head_conductance_completeness_residual.abs()
print(
    f"Completeness residual: largest {float(residual.max()):.4f} at layer {int(residual.argmax())} "
    f"({conductance_steps} steps)"
)
print(
    f"Target (Austin - Denver logit): Dallas prompt {float(conductance.head_conductance_t_from):.4f}, "
    f"Denver prompt {float(conductance.head_conductance_t_to):.4f}"
)
print(
    f"Measured change: {float(conductance.head_conductance_t_to) - float(conductance.head_conductance_t_from):.4f}; "
    f"first-order prediction at the Dallas prompt: {float(conductance.head_conductance_first_order):.4f}"
)
assert float(conductance.head_conductance_t_from) != float(conductance.head_conductance_t_to)

for name, table in (("pattern (QK)", conductance.head_conductance_qk), ("value (OV)", conductance.head_conductance_ov)):
    flat = table.abs().flatten().topk(5)
    heads = [(int(i) // table.shape[1], int(i) % table.shape[1], float(table.flatten()[int(i)])) for i in flat.indices]
    print(f"Top {name} heads (layer, head, credit): {heads}")
assert float(conductance.head_conductance_qk.abs().max()) > 0 and float(conductance.head_conductance_ov.abs().max()) > 0

## Summary

| Surface | What it showed here |
|----|---------|
| `compute_attribution_graph` regimes | The Austin edges differ between frozen and live patterns, and each graph records which regime built it. A live edge is a first-order derivative, not an exact direct effect. |
| `head_loadings` | The edges into Austin split across heads per layer, with the partition check read beside the shares and the summed edges named as the shares' denominator. |
| `head_conductance` | The Dallas-to-Denver change of the Austin-versus-Denver logit difference, credited to heads in QK and OV parts, with the completeness residual read first. |

Only the nnsight model-level adapter supports this surface for now. The frozen-regime graph machinery behind
the first section is the subject of the [CT analysis backend demo](ct_analysis_backend_demo.ipynb), and
gradient- and ablation-based attribution live in the
[attribution analysis](../attribution_analysis/attribution_analysis.ipynb) notebook.
